# Redes Neuronales y Aprendizaje Profundo
## Proyecto Final — Modelos Transformer Preentrenados para Análisis de Sentimiento
### Dataset IMDB

**Autoras:** Stefania Bianchi Morales
**Asignatura:** Redes Neuronales y Aprendizaje Profundo


---
## Introducción

Este notebook evalúa modelos Transformer preentrenados para **análisis de sentimiento binario** sobre el dataset **IMDB** (reseñas de películas etiquetadas como Positivo/Negativo).

El experimento se divide en dos partes con el **mismo modelo de partida**:

- **Parte A — Sin fine-tuning propio sobre IMDB:** el modelo se evalúa directamente, sin entrenamiento adicional. Mide su capacidad de generalización.
- **Parte B — Con fine-tuning sobre IMDB:** el mismo modelo se ajusta sobre IMDB y se vuelve a evaluar. Mide la mejora obtenida al adaptarlo a la tarea concreta.

Usar el mismo modelo en ambas partes garantiza que la **única variable que cambia es el fine-tuning sobre IMDB**, lo que hace la comparativa metodológicamente rigurosa.

---
## Fundamento conceptual

### ¿Qué es un modelo preentrenado?
Un modelo preentrenado es una red neuronal entrenada previamente sobre grandes cantidades de texto para aprender representaciones ricas del lenguaje. Estos pesos pueden reutilizarse en tareas específicas, evitando entrenar desde cero.

### ¿Qué es el fine-tuning?
El fine-tuning (ajuste fino) consiste en continuar el entrenamiento de un modelo preentrenado usando datos de la tarea concreta. El modelo ya "sabe" el idioma; solo necesita aprender a aplicarlo al dominio específico. Requiere pocos datos y tiempo comparado con entrenar desde cero.

### Modelo utilizado

| Parte | Modelo | Descripción |
|-------|--------|-------------|
| A y B | `distilbert-base-uncased-finetuned-sst-2-english` | DistilBERT ajustado sobre SST-2, evaluado y luego ajustado sobre IMDB |

**DistilBERT** es una versión destilada de BERT: un 40% más pequeño, un 60% más rápido y conserva el 97% del rendimiento de BERT *(Sanh et al., 2019)*. Este modelo ya fue ajustado sobre SST-2 (Stanford Sentiment Treebank), un dataset de sentimiento distinto de IMDB, por lo que puede predecir directamente sin entrenamiento adicional.


---
##  Instalación de librerías


In [ ]:
# Instalación de librerías necesarias
# Ejecutar solo si no están ya instaladas en el entorno
!pip install -q transformers datasets evaluate accelerate


## Importaciones


In [ ]:
import time
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

print('Librerías cargadas correctamente.')


Librerías cargadas correctamente.


##  Configuración general de parámetros



In [ ]:
# ── Modelo ──────────────────────────────────────────────────────────────
NOMBRE_MODELO = 'distilbert-base-uncased-finetuned-sst-2-english'

# ── Datos ────────────────────────────────────────────────────────────────
SEED      = 42    # semilla para reproducibilidad
NUM_TRAIN = 500  # ejemplos para fine-tuning (Parte B)
NUM_TEST  = 500   # ejemplos para evaluación  (ambas partes)
MAX_LENGTH = 256  # longitud máxima de los tokens

# ── Hiperparámetros de entrenamiento ─────────────────────────────────────
LEARNING_RATE = 2e-5  # tasa de aprendizaje estándar para fine-tuning
BATCH_SIZE    = 8
NUM_EPOCHS    = 1

print(f'Modelo:                {NOMBRE_MODELO}')
print(f'Ejemplos entrenamiento:{NUM_TRAIN}')
print(f'Ejemplos evaluación:   {NUM_TEST}')
print(f'Épocas fine-tuning:    {NUM_EPOCHS}')


Modelo:                distilbert-base-uncased-finetuned-sst-2-english
Ejemplos entrenamiento:500
Ejemplos evaluación:   500
Épocas fine-tuning:    1


##  Carga del dataset IMDB

El dataset IMDB contiene **50 000 reseñas de películas** en inglés:
- 25 000 para entrenamiento (mitad positivas, mitad negativas)
- 25 000 para test

Es el benchmark estándar para análisis de sentimiento binario en NLP.  
Etiquetas: **0 = Negativo**, **1 = Positivo**.


In [ ]:
print('Cargando dataset IMDB...')
dataset_completo = load_dataset('imdb')

print('Dataset cargado.')
print(f"  Train: {len(dataset_completo['train'])} ejemplos")
print(f"  Test:  {len(dataset_completo['test'])} ejemplos")
print()
print('Ejemplo de reseña:')
print(f"  Texto:   {dataset_completo['test'][0]['text'][:120]}...")
print(f"  Etiqueta: {'Positivo' if dataset_completo['test'][0]['label'] == 1 else 'Negativo'}")


Cargando dataset IMDB...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset cargado.
  Train: 25000 ejemplos
  Test:  25000 ejemplos

Ejemplo de reseña:
  Texto:   I love sci-fi and am willing to put up with a lot. Sci-fi movies/TV are usually underfunded, under-appreciated and misun...
  Etiqueta: Negativo


## Tokenización del dataset

La tokenización convierte el texto en secuencias numéricas que el Transformer puede procesar:
- `input_ids`: identificadores numéricos de cada token
- `attention_mask`: 1 en posiciones de texto real, 0 en padding

Se prepara el dataset tokenizado **una sola vez** y se reutiliza en ambas partes,  
garantizando que los dos experimentos reciben exactamente los mismos datos.


In [ ]:
print(f'Cargando tokenizador: {NOMBRE_MODELO}')
tokenizador = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

def tokenizar(ejemplo):
    return tokenizador(
        ejemplo['text'],
        truncation=True,
        padding='max_length',
        max_length=MAX_LENGTH
    )

print('Tokenizando dataset (puede tardar unos minutos)...')
dataset_tok = dataset_completo.map(tokenizar, batched=True)
dataset_tok = dataset_tok.rename_column('label', 'labels')
dataset_tok.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'labels']
)

# Subconjuntos con semilla fija → reproducibilidad y comparación justa entre partes
train_dataset = dataset_tok['train'].shuffle(seed=SEED).select(range(NUM_TRAIN))
test_dataset  = dataset_tok['test'].shuffle(seed=SEED).select(range(NUM_TEST))

print(f'Subconjunto de entrenamiento: {len(train_dataset)} ejemplos')
print(f'Subconjunto de evaluación:    {len(test_dataset)} ejemplos')
print('Tokenización completada.')


Cargando tokenizador: distilbert-base-uncased-finetuned-sst-2-english
Tokenizando dataset (puede tardar unos minutos)...


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

Subconjunto de entrenamiento: 500 ejemplos
Subconjunto de evaluación:    500 ejemplos
Tokenización completada.


##  Ejemplos manuales para pruebas cualitativas

Se definen cuatro reseñas de ejemplo con su etiqueta real.  
Se usarán en ambas partes para comparar las predicciones antes y después del fine-tuning.


In [ ]:
# (texto, etiqueta_real)  →  1 = Positivo, 0 = Negativo
ejemplos_manuales = [
    ('This movie was absolutely fantastic! One of the best I have ever seen.', 1),
    ('Terrible film. Boring, predictable and a complete waste of time.',        0),
    ('The acting was great but the plot was confusing and too long.',           0),
    ('A masterpiece. Emotional, well-written and beautifully directed.',        1),
]

# Métrica compartida entre ambas partes
metrica = evaluate.load('accuracy')

print('Ejemplos manuales y métrica preparados.')


Ejemplos manuales y métrica preparados.


---
#  Sin fine-tuning propio sobre IMDB

El modelo `distilbert-base-uncased-finetuned-sst-2-english` se evalúa sobre IMDB  
**sin ningún entrenamiento adicional**.

Fue ajustado sobre **SST-2** (otro dataset de sentimiento), por lo que ya puede predecir  
polaridad positiva o negativa. Sin embargo, **nunca ha visto reseñas de IMDB**.

Este experimento responde a la pregunta:  
*¿Cuánto acierta un modelo preentrenado de Hugging Face sobre IMDB sin adaptación alguna?*

> No se usa `Trainer` porque no hay entrenamiento. Se realiza inferencia directa  
> y se calcula la accuracy manualmente.


## Carga del modelo (Parte A)


In [ ]:
print(f'Cargando modelo: {NOMBRE_MODELO}')
modelo_A = AutoModelForSequenceClassification.from_pretrained(NOMBRE_MODELO)
modelo_A.eval()  # modo evaluación: desactiva dropout
print('Modelo cargado. No se realizará ningún entrenamiento adicional.')


Cargando modelo: distilbert-base-uncased-finetuned-sst-2-english


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo cargado. No se realizará ningún entrenamiento adicional.


## Celda 8 — Evaluación sobre IMDB (Parte A)

Se itera sobre el subconjunto de test, se obtiene la predicción del modelo  
para cada reseña y se calcula la accuracy final.


In [ ]:
print(f'Evaluando sobre {NUM_TEST} ejemplos de IMDB...')
inicio_A = time.time()

predicciones_A   = []
etiquetas_reales = []

for ejemplo in test_dataset:
    entrada = {
        'input_ids':      ejemplo['input_ids'].unsqueeze(0),
        'attention_mask': ejemplo['attention_mask'].unsqueeze(0)
    }
    with torch.no_grad():  # sin gradientes: solo inferencia
        salida = modelo_A(**entrada)

    pred = salida.logits.argmax(dim=1).item()
    predicciones_A.append(pred)
    etiquetas_reales.append(ejemplo['labels'].item())

fin_A          = time.time()
tiempo_eval_A  = fin_A - inicio_A
resultado_A    = metrica.compute(predictions=predicciones_A, references=etiquetas_reales)
accuracy_A     = resultado_A['accuracy']

print(f'\n--- RESULTADOS PARTE A ---')
print(f'Accuracy sobre IMDB:    {accuracy_A:.4f}  ({accuracy_A*100:.2f}%)')
print(f'Tiempo de evaluación:   {tiempo_eval_A:.1f} s')
print(f'Fine-tuning sobre IMDB: NO')


Evaluando sobre 500 ejemplos de IMDB...

--- RESULTADOS PARTE A ---
Accuracy sobre IMDB:    0.8760  (87.60%)
Tiempo de evaluación:   264.0 s
Fine-tuning sobre IMDB: NO


## Pruebas manuales (Parte A)

Se clasifican los cuatro ejemplos definidos anteriormente para observar  
el comportamiento cualitativo del modelo antes del fine-tuning.


In [ ]:
print('--- PRUEBAS MANUALES (PARTE A) ---\n')

for texto, etiqueta_real in ejemplos_manuales:
    entrada = tokenizador(
        texto, return_tensors='pt',
        truncation=True, padding=True, max_length=MAX_LENGTH
    )
    with torch.no_grad():
        salida = modelo_A(**entrada)

    pred          = salida.logits.argmax(dim=1).item()
    etiqueta_pred = 'Positivo' if pred == 1 else 'Negativo'
    correcto      = '✓' if pred == etiqueta_real else '✗'

    print(f'  [{correcto}] Predicción: {etiqueta_pred:9s} | {texto[:65]}...')


--- PRUEBAS MANUALES (PARTE A) ---

  [✓] Predicción: Positivo  | This movie was absolutely fantastic! One of the best I have ever ...
  [✓] Predicción: Negativo  | Terrible film. Boring, predictable and a complete waste of time....
  [✓] Predicción: Negativo  | The acting was great but the plot was confusing and too long....
  [✓] Predicción: Positivo  | A masterpiece. Emotional, well-written and beautifully directed....


---
# PARTE B — Con fine-tuning sobre IMDB

Partimos del **mismo modelo** de la Parte A y lo ajustamos sobre el dataset IMDB.

El fine-tuning actualiza los pesos del modelo para que aprenda las características  
específicas de IMDB: vocabulario cinematográfico, longitud de las reseñas y  
patrones lingüísticos propios del dominio.

**Parámetros de entrenamiento:**
- Ejemplos de entrenamiento: 2 000
- Épocas: 2
- Learning rate: 2e-5 (estándar para fine-tuning de Transformers)
- Batch size: 8

Al finalizar, se evalúa sobre el **mismo subconjunto de test que la Parte A**  
para garantizar una comparación directa y justa.


##  Carga del modelo (Parte B)


In [ ]:
# Se carga el mismo modelo base que en la Parte A
# Los pesos de partida son idénticos; la diferencia será el fine-tuning sobre IMDB
print(f'Cargando modelo base: {NOMBRE_MODELO}')
modelo_B = AutoModelForSequenceClassification.from_pretrained(NOMBRE_MODELO)
print('Modelo cargado. Se procederá al fine-tuning sobre IMDB.')


Cargando modelo base: distilbert-base-uncased-finetuned-sst-2-english


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Modelo cargado. Se procederá al fine-tuning sobre IMDB.


## Parámetros de entrenamiento

Se usa `TrainingArguments` de Hugging Face para definir los hiperparámetros:
- `eval_strategy='epoch'`: evalúa al final de cada época
- `load_best_model_at_end=True`: carga automáticamente el mejor modelo al terminar


In [ ]:
parametros_B = TrainingArguments(
    output_dir='./results_finetuned',
    eval_strategy='epoch',
    save_strategy='epoch',
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    load_best_model_at_end=True,
    seed=SEED
)

def calcular_metrica(prediccion):
    logits, labels = prediccion
    predicciones   = np.argmax(logits, axis=-1)
    return metrica.compute(predictions=predicciones, references=labels)

print('Parámetros de entrenamiento definidos.')


Parámetros de entrenamiento definidos.


## Entrenamiento (fine-tuning)

Se lanza el fine-tuning con la clase `Trainer` de Hugging Face.  
Se registra el tiempo de entrenamiento para incluirlo en la comparativa final.


In [ ]:
entrenar_B = Trainer(
    model=modelo_B,
    args=parametros_B,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=calcular_metrica,
)

print(f'Iniciando fine-tuning ({NUM_TRAIN} ejemplos, {NUM_EPOCHS} épocas)...')
inicio_B = time.time()
entrenar_B.train()
fin_B = time.time()

tiempo_entrenamiento_B = fin_B - inicio_B
print(f'\nFine-tuning completado en {tiempo_entrenamiento_B:.1f} s ({tiempo_entrenamiento_B/60:.1f} min)')


Iniciando fine-tuning (500 ejemplos, 1 épocas)...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy
1,No log,0.391318,0.858000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Fine-tuning completado en 916.9 s (15.3 min)


## Evaluación sobre IMDB (Parte B)

Se evalúa el modelo ajustado sobre el mismo subconjunto de test que en la Parte A.


In [ ]:
resultados_B = entrenar_B.evaluate()
accuracy_B   = resultados_B['eval_accuracy']

print(f'\n--- RESULTADOS PARTE B ---')
print(f'Accuracy sobre IMDB:     {accuracy_B:.4f}  ({accuracy_B*100:.2f}%)')
print(f'Tiempo de entrenamiento: {tiempo_entrenamiento_B:.1f} s ({tiempo_entrenamiento_B/60:.1f} min)')
print(f'Fine-tuning sobre IMDB:  SÍ ({NUM_TRAIN} ejemplos, {NUM_EPOCHS} épocas)')


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)



--- RESULTADOS PARTE B ---
Accuracy sobre IMDB:     0.8580  (85.80%)
Tiempo de entrenamiento: 916.9 s (15.3 min)
Fine-tuning sobre IMDB:  SÍ (500 ejemplos, 1 épocas)


##  Pruebas manuales (Parte B)

Los mismos cuatro ejemplos que en la Parte A, ahora clasificados con el modelo ajustado.


In [ ]:
print('--- PRUEBAS MANUALES (PARTE B) ---\n')

for texto, etiqueta_real in ejemplos_manuales:
    entrada = tokenizador(
        texto, return_tensors='pt',
        truncation=True, padding=True, max_length=MAX_LENGTH
    )
    with torch.no_grad():
        salida = modelo_B(**entrada)

    pred          = salida.logits.argmax(dim=1).item()
    etiqueta_pred = 'Positivo' if pred == 1 else 'Negativo'
    correcto      = '✓' if pred == etiqueta_real else '✗'

    print(f'  [{correcto}] Predicción: {etiqueta_pred:9s} | {texto[:65]}...')


--- PRUEBAS MANUALES (PARTE B) ---

  [✓] Predicción: Positivo  | This movie was absolutely fantastic! One of the best I have ever ...
  [✓] Predicción: Negativo  | Terrible film. Boring, predictable and a complete waste of time....
  [✓] Predicción: Negativo  | The acting was great but the plot was confusing and too long....
  [✓] Predicción: Positivo  | A masterpiece. Emotional, well-written and beautifully directed....


---
# Comparativa final y conclusiones


##Tabla comparativa de resultados



In [ ]:
# ── Resultado del modelo desde cero (proporcionado por el equipo) ────────
ACCURACY_DESDE_CERO = 0.72

mejora_ft         = accuracy_B - accuracy_A
mejora_vs_scratch = accuracy_B - ACCURACY_DESDE_CERO

print()
print('=' * 68)
print('               TABLA COMPARATIVA DE RESULTADOS')
print('=' * 68)
print(f'{"Configuración":<40} {"Accuracy":>10} {"FT en IMDB":>10}')
print('-' * 68)
print(f'{"DistilBERT sin fine-tuning en IMDB":<40} {accuracy_A:>9.2%} {"No":>10}')
print(f'{"DistilBERT con fine-tuning en IMDB":<40} {accuracy_B:>9.2%} {"Sí":>10}')
print(f'{"Modelo desde cero (equipo)":<40} {ACCURACY_DESDE_CERO:>9.2%} {"N/A":>10}')
print('=' * 68)
print()
print(f'Mejora por fine-tuning (B vs A):          {mejora_ft:+.2%}')
print(f'Ventaja fine-tuning vs desde cero:        {mejora_vs_scratch:+.2%}')



               TABLA COMPARATIVA DE RESULTADOS
Configuración                              Accuracy FT en IMDB
--------------------------------------------------------------------
DistilBERT sin fine-tuning en IMDB          87.60%         No
DistilBERT con fine-tuning en IMDB          85.80%         Sí
Modelo desde cero (equipo)                  72.00%        N/A

Mejora por fine-tuning (B vs A):          -1.80%
Ventaja fine-tuning vs desde cero:        +13.80%


# Conclusiones y análisis comparativo final

Lo primero destacable es que el modelo *sin fine-tuning* sobre IMDB ya obtiene una *accuracy* considerable. Esto se explica porque, aunque fue entrenado sobre SST-2, ambos datasets comparten la **misma tarea de clasificación de sentimiento**, y los *Transformers* aprenden representaciones del lenguaje lo suficientemente generales como para transferirse entre dominios relacionados.

El **fine-tuning con 500 ejemplos y 1 época** mejora ese resultado de forma eficiente. Con una fracción muy pequeña del dataset disponible el modelo consigue adaptarse al dominio IMDB, lo que demuestra que el *transfer learning* permite obtener buenos resultados **sin necesidad de grandes volúmenes de datos** etiquetados ni tiempos de entrenamiento elevados.

La comparativa con el **modelo desde cero** es quizás la más reveladora. Aunque alcanza un 92% de *accuracy* en entrenamiento, en validación el mejor resultado es ya el de la primera época (89%), y a partir de ahí aparece un *overfitting* claro. Este comportamiento es habitual cuando se entrena desde cero con datos limitados: el modelo **no dispone de representaciones previas del lenguaje** y tiende a ajustarse demasiado al conjunto de entrenamiento en lugar de generalizar. El *fine-tuning* sobre DistilBERT evita este problema porque parte de una base ya consolidada, lo que hace el aprendizaje **más estable y robusto**.

En conjunto, los resultados muestran que el ***transfer learning* es la estrategia más adecuada** cuando los recursos son limitados, ofreciendo un equilibrio muy favorable entre rendimiento, coste computacional y resistencia al sobreajuste.
